# 03b — Churn Label Audit & Corrected Export (v6)
SpiriCom · Huawei Technologies Tunisia · PFE 2026

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime

pd.set_option('display.float_format', '{:.4f}'.format)
PROC_DIR  = Path('data/processed')
OUT_DIR   = Path('data/outputs')
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df_kpi  = pd.read_parquet(PROC_DIR / 'kpi_clean.parquet')
df_kpi.columns = df_kpi.columns.str.lower().str.replace(' ', '_')
df_comp = pd.read_parquet(PROC_DIR / 'complaints_clean.parquet')
df_comp.columns = df_comp.columns.str.lower().str.replace(' ', '_')
print(f'KPI        : {df_kpi.shape[0]:,} rows x {df_kpi.shape[1]} cols')
print(f'Complaints : {df_comp.shape[0]:,} rows x {df_comp.shape[1]} cols')


KPI        : 4,898 rows x 122 cols
Complaints : 25,727 rows x 23 cols


## 1 — Imputation forensics (NB03-2)
NB00 filled nulls with the column median, destroying the missingness information.
A median-imputed column shows a **value spike**: one exact value repeated far more
often than any continuous variable allows. We flag those rows.

> Better long-term fix: re-export `kpi_clean.parquet` from NB00 **with NaNs kept**
> (or with explicit `*_was_null` flags). The heuristic below is the recovery path
> when the raw NaN mask is no longer available.

In [2]:
IMPUTE_CHECK_COLS = ['dou_total', 'duration', 'traffic_5g',
                     'traffic_4g', 'traffic_3g', 'traffic_2g',
                     'e2e_delay_ms', 'client_rtt_ms']
IMPUTE_CHECK_COLS = [c for c in IMPUTE_CHECK_COLS if c in df_kpi.columns]

SPIKE_FRAC = 0.05   # a single exact value covering >5% of a continuous column
imputed_flags = {}
print(f'{"column":<22s} {"spike value":>20s} {"spike rows":>11s} {"share":>7s}')
for c in IMPUTE_CHECK_COLS:
    vc = df_kpi[c].value_counts()
    if len(vc) == 0:
        continue
    spike_val, spike_n = vc.index[0], int(vc.iloc[0])
    share = spike_n / len(df_kpi)
    flagged = share > SPIKE_FRAC
    if flagged:
        imputed_flags[c] = spike_val
        df_kpi[f'{c}_imputed'] = (df_kpi[c] == spike_val).astype(int)
    print(f'{c:<22s} {spike_val:>20,.0f} {spike_n:>11,} {share*100:>6.1f}%'
          + ('  <- IMPUTED (flagged)' if flagged else ''))

print(f'\nColumns with detected median-imputation: {list(imputed_flags)}')
if 'dou_total' in imputed_flags or 'duration' in imputed_flags:
    print('CONFIRMED NB03-2: label-defining columns were imputed in NB00.')
    print('These rows cannot be labelled honestly - they will get churn = NaN.')


column                          spike value  spike rows   share
dou_total                       231,108,281       1,358   27.7%  <- IMPUTED (flagged)
duration                                353       1,509   30.8%  <- IMPUTED (flagged)
traffic_5g                          193,498       4,498   91.8%  <- IMPUTED (flagged)
traffic_4g                      254,279,290       1,734   35.4%  <- IMPUTED (flagged)
traffic_3g                        3,746,973       2,028   41.4%  <- IMPUTED (flagged)
traffic_2g                           65,048       3,940   80.4%  <- IMPUTED (flagged)
e2e_delay_ms                            164       2,072   42.3%  <- IMPUTED (flagged)
client_rtt_ms                           118       2,092   42.7%  <- IMPUTED (flagged)

Columns with detected median-imputation: ['dou_total', 'duration', 'traffic_5g', 'traffic_4g', 'traffic_3g', 'traffic_2g', 'e2e_delay_ms', 'client_rtt_ms']
CONFIRMED NB03-2: label-defining columns were imputed in NB00.
These rows cannot be labelle

## 2 — MSISDN linkage rescue attempt (NB03-3)

In [3]:
def norm_variants(s):
    s = s.dropna().astype(str).str.strip()
    s = s[s.str.len() >= 4]                      # drop garbage '', '0', '00'
    digits = s.str.replace(r'\D', '', regex=True)
    return {
        'raw'        : set(s),
        'lstrip_zero': set(s.str.lstrip('0')),
        'last8'      : set(digits.str[-8:]),
        'digits_only': set(digits),
    }

v_comp = norm_variants(df_comp['msisdn'])
v_kpi  = norm_variants(df_kpi['msisdn'])

print(f'{"strategy":<14s} {"comp":>8s} {"kpi":>8s} {"overlap":>9s} {"rate":>7s}')
best_strategy, best_overlap = None, -1
for k in v_comp:
    ov = len(v_comp[k] & v_kpi[k])
    rate = ov / max(len(v_kpi[k]), 1)
    print(f'{k:<14s} {len(v_comp[k]):>8,} {len(v_kpi[k]):>8,} {ov:>9,} {rate*100:>6.1f}%')
    if ov > best_overlap:
        best_strategy, best_overlap = k, ov

LINKABLE = best_overlap >= 0.05 * df_kpi['msisdn'].nunique()
print(f'\nBest strategy: {best_strategy} ({best_overlap:,} matches)')
if LINKABLE:
    print('Datasets ARE linkable after normalization - complaint features can be')
    print('engineered in NB04 using this strategy.')
else:
    print('CONFIRMED: datasets are not linkable (likely different anonymization')
    print('salts). The spec\'s complaint-feature category is formally dropped;')
    print('complaints analysis stays a standalone deliverable (NB01).')


strategy           comp      kpi   overlap    rate
raw              22,125    4,890         1    0.0%
lstrip_zero      22,085    4,890         1    0.0%
last8            22,123    4,890         1    0.0%
digits_only      22,125    4,890         1    0.0%

Best strategy: raw (1 matches)
CONFIRMED: datasets are not linkable (likely different anonymization
salts). The spec's complaint-feature category is formally dropped;
complaints analysis stays a standalone deliverable (NB01).


## 3 — Label v6: thresholds on observed data, imputed rows unlabelled

In [4]:
agg = dict(
    session_flag=('session_flag', 'max'),
    traffic_5g=('traffic_5g', 'sum'),
    dou_total=('dou_total', 'sum'),
    duration=('duration', 'sum'),
    brand=('brand', 'first'),
)
for c in imputed_flags:
    if f'{c}_imputed' in df_kpi.columns:
        agg[f'{c}_imputed'] = (f'{c}_imputed', 'max')

churn = df_kpi.groupby('msisdn').agg(**agg).reset_index()

dou_imp = churn.get('dou_total_imputed', pd.Series(0, index=churn.index))
dur_imp = churn.get('duration_imputed',  pd.Series(0, index=churn.index))
label_observed = (dou_imp == 0) & (dur_imp == 0)
n_obs = int(label_observed.sum())
print(f'Customers total                  : {len(churn):,}')
print(f'Label-observable (no imputation '
      f'on dou/duration)                 : {n_obs:,} '
      f'({n_obs/len(churn)*100:.1f}%)')

# NB03-2: thresholds from OBSERVED customers only
obs = churn[label_observed]
thr_dou = obs['dou_total'].quantile(0.20)
thr_dur = obs['duration'].quantile(0.20)
print(f'\nDOU Q20 (observed only) : {thr_dou:>15,.0f} bytes ({thr_dou/1e6:.2f} MB)')
print(f'Dur Q20 (observed only) : {thr_dur:>15,.0f} seconds')

churn['c1_low_usage'] = (churn['dou_total'] <= thr_dou).astype(int)
churn['c2_low_dur']   = (churn['duration']  <= thr_dur).astype(int)
churn['churn'] = np.where(
    label_observed,
    ((churn['c1_low_usage'] == 1) | (churn['c2_low_dur'] == 1)).astype(float),
    np.nan,                                  # imputed -> unlabelled
)

lab = churn['churn'].dropna()
rate = lab.mean()
print(f'\n=== DISENGAGEMENT LABEL v6 ===')
print(f'Labelled population : {len(lab):,}')
print(f'Disengaged          : {int(lab.sum()):,}')
print(f'Engaged             : {int((lab==0).sum()):,}')
print(f'Disengaged share    : {rate*100:.1f}%  '
      '(design parameter Q20+OR, not a measured churn rate - NB03-4)')
print(f'Unlabelled (imputed): {int(churn["churn"].isna().sum()):,}')

# NB03-5: structural assertions only
assert len(churn) > 0 and len(lab) > 0
assert thr_dou > 0 and thr_dur > 0
assert 0 < rate < 1
print('\nStructural assertions passed')


Customers total                  : 4,896
Label-observable (no imputation on dou/duration)                 : 2,566 (52.4%)

DOU Q20 (observed only) :       1,856,088 bytes (1.86 MB)
Dur Q20 (observed only) :              82 seconds

=== DISENGAGEMENT LABEL v6 ===
Labelled population : 2,566
Disengaged          : 868
Engaged             : 1,698
Disengaged share    : 33.8%  (design parameter Q20+OR, not a measured churn rate - NB03-4)
Unlabelled (imputed): 2,330

Structural assertions passed


## 4 — Leakage blocklist (NB03-1)
Everything the label is computed from, plus every additive component of it.
NB04/NB05 **must** load this file and drop these columns from the feature set.
What remains is the genuinely interesting model: *network experience + device +
geography -> disengagement*.

In [5]:
VOLUME_FAMILY = [c for c in df_kpi.columns if c.endswith('_traffic')]
LEAKAGE_BLOCKLIST = sorted(set(
    ['dou_total', 'duration', 'session_flag',
     'traffic_2g', 'traffic_3g', 'traffic_4g', 'traffic_5g',
     'voice_onlinetime_2g', 'voice_onlinetime_3g',
     'night_traffic', 'day_traffic', 'late_night_traffic',
     'c1_low_usage', 'c2_low_dur']
    + VOLUME_FAMILY
) & set(list(df_kpi.columns) + ['c1_low_usage', 'c2_low_dur']))

ALLOWED_EXAMPLES = [c for c in [
    'e2e_delay_ms', 'client_rtt_ms', 'server_rtt_ms',
    'client_packet_loss_rate', 'server_packet_loss_rate',
    'page_response_delay', 'page_download_throughput',
    'dns_delay', 'dns_sr', 'tcp_connection_sr',
    'brand', 'generation', 'tertype', 'sim_capability', 'usim_flag',
    'usertype', 'user_class', 'mobility_class', 'number_of_regions',
    'layer2name', 'layer3name',
] if c in df_kpi.columns]

blocklist_payload = {
    'generated_at': datetime.now().isoformat(),
    'reason': 'churn v6 label is a deterministic function of dou_total/duration; '
              'all volume and duration features leak the label (NB03-1)',
    'blocked_features': LEAKAGE_BLOCKLIST,
    'allowed_examples': ALLOWED_EXAMPLES,
    'thresholds': {'dou_q20_bytes': float(thr_dou),
                   'dur_q20_seconds': float(thr_dur)},
}
bl_path = MODEL_DIR / 'leakage_blocklist.json'
json.dump(blocklist_payload, open(bl_path, 'w'), indent=2)
print(f'Saved: {bl_path}')
print(f'Blocked ({len(LEAKAGE_BLOCKLIST)}): {LEAKAGE_BLOCKLIST}')
print(f'\nAllowed examples ({len(ALLOWED_EXAMPLES)}): {ALLOWED_EXAMPLES}')


Saved: models\leakage_blocklist.json
Blocked (31): ['c1_low_usage', 'c2_low_dur', 'day_traffic', 'dou_total', 'duration', 'facebook_messenger_traffic', 'facebook_traffic', 'game_traffic', 'google_common_traffic', 'googlesearch_traffic', 'https_traffic', 'im_traffic', 'instagram_traffic', 'late_night_traffic', 'night_traffic', 'other_traffic', 'quic_ietf_traffic', 'session_flag', 'sms_traffic', 'streaming_traffic', 'tiktok_traffic', 'traffic_2g', 'traffic_3g', 'traffic_4g', 'traffic_5g', 'voice_onlinetime_2g', 'voice_onlinetime_3g', 'voip_traffic', 'web_browsing_traffic', 'whatsapp_traffic', 'youtube_traffic']

Allowed examples (20): ['e2e_delay_ms', 'client_rtt_ms', 'server_rtt_ms', 'client_packet_loss_rate', 'server_packet_loss_rate', 'page_response_delay', 'page_download_throughput', 'dns_delay', 'dns_sr', 'tcp_connection_sr', 'brand', 'generation', 'tertype', 'usim_flag', 'usertype', 'user_class', 'mobility_class', 'number_of_regions', 'layer2name', 'layer3name']


## 5 — Export `churn_labelled_v6.parquet` + corrected JSON

In [6]:
# Attach modelling-safe context columns
for col in ['generation', 'layer2name', 'mobility_class', 'usertype',
            'user_class', 'tertype']:
    if col in df_kpi.columns and col not in churn.columns:
        m = df_kpi.drop_duplicates('msisdn')[['msisdn', col]].copy()
        m[col] = (m[col].astype(str).str.strip().str.upper()
                  .replace({'NAN': 'UNKNOWN', '': 'UNKNOWN'}))
        churn = churn.merge(m, on='msisdn', how='left')

QOS_COLS = [c for c in [
    'e2e_delay_ms', 'client_rtt_ms', 'server_rtt_ms',
    'client_packet_loss_rate', 'server_packet_loss_rate',
    'page_response_delay', 'page_download_throughput',
    'dns_delay', 'dns_sr', 'tcp_connection_sr', 'number_of_regions',
] if c in df_kpi.columns]
if QOS_COLS:
    qos = df_kpi.groupby('msisdn')[QOS_COLS].mean().reset_index()
    churn = churn.merge(qos, on='msisdn', how='left')

out_path = PROC_DIR / 'churn_labelled_v6.parquet'
churn.to_parquet(out_path, index=False)
print(f'Saved: {out_path}  ({churn.shape[0]:,} rows x {churn.shape[1]} cols)')
print('churn column: 1.0 disengaged / 0.0 engaged / NaN unlabelled (imputed)')

lab = churn['churn'].dropna()
eda_json = {
    'generated_at'        : datetime.now().isoformat(),
    'label_version'       : 'v6 - observed-only thresholds, imputed rows unlabelled',
    'definition_note'     : 'disengagement segment (dou<=Q20 OR duration<=Q20), '
                            'a design parameter - NOT a measured churn rate',
    'total_customers'     : int(len(churn)),
    'labelled_customers'  : int(len(lab)),
    'unlabelled_imputed'  : int(churn['churn'].isna().sum()),
    'disengaged'          : int(lab.sum()),
    'engaged'             : int((lab == 0).sum()),
    'disengaged_share_pct': round(float(lab.mean()) * 100, 2),
    'thresholds'          : {'dou_q20_bytes': int(thr_dou),
                             'dur_q20_seconds': int(thr_dur)},
    'msisdn_linkage'      : {'best_strategy': best_strategy,
                             'best_overlap': int(best_overlap),
                             'linkable': bool(LINKABLE)},
    'leakage_blocklist'   : str(bl_path),
    'saved_file'          : str(out_path),
}
jp = OUT_DIR / 'churn_eda_v6.json'
json.dump(eda_json, open(jp, 'w'), indent=2, default=str)
print(f'Saved: {jp}')
print(json.dumps(eda_json, indent=2, default=str))
print('\nNext -> NB04 must: load churn_labelled_v6.parquet, drop NaN labels,')
print('apply models/leakage_blocklist.json, and engineer features ONLY from')
print('the allowed families (QoS / device / geography / mobility).')


Saved: data\processed\churn_labelled_v6.parquet  (4,896 rows x 34 cols)
churn column: 1.0 disengaged / 0.0 engaged / NaN unlabelled (imputed)
Saved: data\outputs\churn_eda_v6.json
{
  "generated_at": "2026-06-24T13:28:57.131502",
  "label_version": "v6 - observed-only thresholds, imputed rows unlabelled",
  "definition_note": "disengagement segment (dou<=Q20 OR duration<=Q20), a design parameter - NOT a measured churn rate",
  "total_customers": 4896,
  "labelled_customers": 2566,
  "unlabelled_imputed": 2330,
  "disengaged": 868,
  "engaged": 1698,
  "disengaged_share_pct": 33.83,
  "thresholds": {
    "dou_q20_bytes": 1856088,
    "dur_q20_seconds": 82
  },
  "msisdn_linkage": {
    "best_strategy": "raw",
    "best_overlap": 1,
    "linkable": false
  },
  "leakage_blocklist": "models\\leakage_blocklist.json",
  "saved_file": "data\\processed\\churn_labelled_v6.parquet"
}

Next -> NB04 must: load churn_labelled_v6.parquet, drop NaN labels,
apply models/leakage_blocklist.json, and en